# Kaggle Test Evaluation

Notebook nay evaluate cac checkpoint da train tren tap `test-00000-of-00001.parquet` ngay tren Kaggle.

Notebook tu tai output tu kernel `anhnguyenphi/nlp-aio`, gom cac run trong `report_experiment_outputs`/`summarization_outputs`, roi chia run thanh 2 shard de chay song song tren T4x2.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True
WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
RUNS_ROOT = WORKING / 'test_eval_runs'
KERNEL_OUTPUT_DIR = WORKING / 'kernel_outputs'
KAGGLE_KERNEL_OUTPUTS = [
    'anhnguyenphi/nlp-aio',
]
DOWNLOAD_KERNEL_OUTPUTS = True
MAX_DOWNLOAD_ATTEMPTS = 2
PARALLEL_EVAL = True
EVAL_BATCH_SIZE = 2  # tang len 4 neu T4 khong OOM
OUT_DIR = WORKING / 'test_eval_outputs'
TEST_FILE = Path('/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/test-00000-of-00001.parquet')
TEST_BASENAME = TEST_FILE.name
MAX_TEST_SAMPLES = None  # dat 50 de smoke test nhanh

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(cmd, shell=True, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\nLast lines:\n{"".join(lines[-100:])}')
    return code

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

WORKING.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if REFRESH_REPO and WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)
if not is_repo(WORKING_REPO):
    run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
else:
    run('git pull --ff-only', cwd=WORKING_REPO, check=False)
repo = WORKING_REPO
run('git log --oneline -1', cwd=repo)


In [ ]:
os.chdir(repo)
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.51.0,<5" "tokenizers>=0.22.0,<=0.23.0"', cwd=repo)
run(f'{sys.executable} -m pip check', cwd=repo, check=False)
run(f'{sys.executable} -m pip show transformers tokenizers peft accelerate | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)
run(f"{sys.executable} -c 'import tokenizers, transformers; print(\"TRANSFORMERS\", transformers.__version__); print(\"TOKENIZERS\", tokenizers.__version__)'", cwd=repo)
run('nvidia-smi', check=False)


## Prepare Runs And Test File

Notebook tai 2 kernel output, xu ly dung 2 layout hien co: `summarization_outputs` cua notebook pretrained/causal va `report_experiment_outputs` cua notebook all-in-one. Moi run co `resolved_config.json` va `best/*.safetensors` se duoc gom vao `/kaggle/working/test_eval_runs`.


In [ ]:
EXPECTED_OUTPUT_ROOT_NAMES = ['summarization_outputs', 'report_experiment_outputs']
RESULT_ZIP_NAMES = ['allinone_report_results.zip', 'summarization_results.zip', 'causal_lm_results.zip']

def find_files(root, pattern):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(root.rglob(pattern))

def download_kernel_outputs():
    if not DOWNLOAD_KERNEL_OUTPUTS:
        return
    KERNEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for kernel in KAGGLE_KERNEL_OUTPUTS:
        dest = KERNEL_OUTPUT_DIR / kernel.split('/')[-1]
        dest.mkdir(parents=True, exist_ok=True)
        done_marker = dest / '.download_complete'
        if done_marker.exists():
            print('KERNEL OUTPUT COMPLETE:', kernel, dest)
            continue
        for attempt in range(1, MAX_DOWNLOAD_ATTEMPTS + 1):
            print(f'DOWNLOAD ATTEMPT {attempt}/{MAX_DOWNLOAD_ATTEMPTS}:', kernel)
            code = run(f'kaggle kernels output {kernel} -p {dest}', cwd=WORKING, check=False)
            if code == 0:
                done_marker.write_text('ok', encoding='utf-8')
                break
            print('WARN: could not fully download kernel output:', kernel)
        if not done_marker.exists():
            print('WARN: using partial output if usable:', dest)

def unzip_result_zips():
    zip_candidates = []
    search_roots = [KERNEL_OUTPUT_DIR, Path('/kaggle/input'), Path('/kaggle/working')]
    for root in search_roots:
        if not root.exists():
            continue
        for name in RESULT_ZIP_NAMES:
            zip_candidates.extend(root.rglob(name))
        zip_candidates.extend(p for p in root.rglob('*.zip') if p.name.startswith(('allinone_', 'summarization_', 'causal_lm_')))
    zip_candidates = sorted(set(p for p in zip_candidates if p.name != 'test_eval_results.zip'))
    if not zip_candidates:
        print('No result zip found. Will try direct kernel output folders.')
        return
    for zip_path in zip_candidates:
        print('UNZIP', zip_path, '->', WORKING)
        with ZipFile(zip_path) as zf:
            zf.extractall(WORKING)

def candidate_output_roots():
    roots = []
    search_roots = [KERNEL_OUTPUT_DIR, WORKING]
    if Path('/kaggle/input').exists():
        search_roots.append(Path('/kaggle/input'))
    for base in search_roots:
        if not base.exists():
            continue
        for root_name in EXPECTED_OUTPUT_ROOT_NAMES:
            direct = base / root_name
            if direct.exists():
                roots.append(direct)
            roots.extend(p for p in base.rglob(root_name) if p.is_dir())
    unique = []
    seen = set()
    for root in sorted(roots):
        key = str(root.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(root)
    return unique

def has_model_artifacts(model_dir):
    return any((model_dir / name).exists() for name in ['adapter_model.safetensors', 'model.safetensors', 'pytorch_model.bin'])

def model_dir_for_run(run_dir):
    best = run_dir / 'best'
    if best.exists() and has_model_artifacts(best):
        return best
    checkpoints = sorted(
        (p for p in run_dir.glob('checkpoint-*') if p.is_dir()),
        key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1,
        reverse=True,
    )
    for checkpoint in checkpoints:
        if has_model_artifacts(checkpoint):
            return checkpoint
    return None

def collect_run_dirs(output_roots):
    found = []
    for root in output_roots:
        for run_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            if (run_dir / 'resolved_config.json').exists() and model_dir_for_run(run_dir) is not None:
                found.append(run_dir)
    unique = []
    seen = set()
    for run_dir in sorted(found):
        key = str(run_dir.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(run_dir)
    return unique

def build_merged_runs_root(run_dirs):
    if RUNS_ROOT.exists():
        shutil.rmtree(RUNS_ROOT)
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    for idx, run_dir in enumerate(run_dirs):
        name = run_dir.name
        dest = RUNS_ROOT / name
        if dest.exists():
            dest = RUNS_ROOT / f'{name}_{idx}'
        try:
            os.symlink(run_dir, dest, target_is_directory=True)
        except OSError:
            shutil.copytree(run_dir, dest)
        print('ADD RUN:', dest.name, '<-', run_dir)

def find_test_file():
    if TEST_FILE.exists():
        return TEST_FILE
    candidates = find_files('/kaggle/input', TEST_BASENAME) + find_files('/kaggle/working', TEST_BASENAME)
    if not candidates:
        raise FileNotFoundError(f'Khong thay {TEST_FILE} hoac file ten {TEST_BASENAME}. Hay attach/upload Kaggle dataset co test parquet.')
    return candidates[0]

download_kernel_outputs()
unzip_result_zips()
output_roots = candidate_output_roots()
print('OUTPUT_ROOT_CANDIDATES:', output_roots)
run_dirs = collect_run_dirs(output_roots)
if not run_dirs:
    raise FileNotFoundError('Khong thay run folder nao co resolved_config.json va best/checkpoint co *.safetensors trong summarization_outputs/report_experiment_outputs. Kernel output co the bi tai dut; hay rerun cell download hoac attach notebook output/dataset day du.')
build_merged_runs_root(run_dirs)
TEST_FILE = find_test_file()
merged_run_dirs = sorted(p for p in RUNS_ROOT.iterdir() if p.is_dir() and (p / 'resolved_config.json').exists())
print('TEST_FILE:', TEST_FILE)
print('RUNS_ROOT:', RUNS_ROOT)
print('RUNS:', [p.name for p in merged_run_dirs])


## Evaluate On Test

Mac dinh chay full test va chia cac run qua 2 GPU neu co T4x2. Neu muon smoke test, sua `MAX_TEST_SAMPLES = 50` o cell dau. Neu muon nhanh hon va khong OOM, tang `EVAL_BATCH_SIZE = 4`.


In [ ]:
import csv
import json
import torch

SHARD_ROOT = WORKING / 'test_eval_shards'
if SHARD_ROOT.exists():
    shutil.rmtree(SHARD_ROOT)
SHARD_ROOT.mkdir(parents=True, exist_ok=True)

all_runs = sorted(p for p in RUNS_ROOT.iterdir() if p.is_dir() and (p / 'resolved_config.json').exists())
if not all_runs:
    raise FileNotFoundError(f'No runs under {RUNS_ROOT}')

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpus = list(range(gpu_count)) if (PARALLEL_EVAL and gpu_count >= 2) else [0 if gpu_count == 1 else None]
print('GPU_COUNT:', gpu_count, 'EVAL_GPUS:', gpus)

shard_roots = []
for shard_idx, gpu in enumerate(gpus):
    shard_root = SHARD_ROOT / f'shard_{shard_idx}'
    shard_root.mkdir(parents=True, exist_ok=True)
    shard_roots.append((shard_root, gpu))

for idx, run_dir in enumerate(all_runs):
    shard_root, _gpu = shard_roots[idx % len(shard_roots)]
    dest = shard_root / run_dir.name
    try:
        os.symlink(run_dir, dest, target_is_directory=True)
    except OSError:
        shutil.copytree(run_dir, dest)
    print('SHARD ADD:', shard_root.name, run_dir.name)

processes = []
for shard_idx, (shard_root, gpu) in enumerate(shard_roots):
    if not any(shard_root.iterdir()):
        continue
    shard_out = OUT_DIR / f'shard_{shard_idx}'
    args = f'--runs_root {shard_root} --test_file {TEST_FILE} --out_dir {shard_out} --eval_batch_size {EVAL_BATCH_SIZE}'
    if MAX_TEST_SAMPLES:
        args += f' --max_test_samples {MAX_TEST_SAMPLES}'
    cmd = f'{sys.executable} -u -m vn_summarization.evaluate_runs_on_test {args}'
    env = os.environ.copy()
    if gpu is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    print('LAUNCH:', cmd, 'CUDA_VISIBLE_DEVICES=', env.get('CUDA_VISIBLE_DEVICES'))
    processes.append((shard_idx, subprocess.Popen(cmd, shell=True, cwd=str(repo), env=env)))

for shard_idx, process in processes:
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'Eval shard {shard_idx} failed with exit code {code}')


def to_float(value):
    try:
        return float(value)
    except Exception:
        return float('-inf')

rows = []
for csv_path in sorted(OUT_DIR.glob('shard_*/test_results.csv')):
    with csv_path.open('r', encoding='utf-8', newline='') as f:
        rows.extend(csv.DictReader(f))
rows.sort(key=lambda row: to_float(row.get('rougeL')), reverse=True)
if not rows:
    raise RuntimeError('No shard test results found')

columns = list(rows[0].keys())
merged_csv = OUT_DIR / 'test_results.csv'
with merged_csv.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

md_columns = ['run', 'kind', 'model', 'rouge1', 'rouge2', 'rougeL', 'loss', 'gen_len']
lines = ['# Test Results', '', '| ' + ' | '.join(md_columns) + ' |', '| ' + ' | '.join(['---'] * len(md_columns)) + ' |']
for row in rows:
    lines.append('| ' + ' | '.join(str(row.get(col, '')) for col in md_columns) + ' |')
lines.append('')
merged_md = OUT_DIR / 'test_results.md'
merged_md.write_text('\n'.join(lines), encoding='utf-8')
(OUT_DIR / 'best_test_run.json').write_text(json.dumps(rows[0], ensure_ascii=False, indent=2), encoding='utf-8')
print(merged_md.read_text(encoding='utf-8'))
print('MERGED CSV:', merged_csv)
print('BEST:', OUT_DIR / 'best_test_run.json')


In [ ]:
zip_path = WORKING / 'test_eval_results.zip'
if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt'}
files = [p for p in OUT_DIR.rglob('*') if p.is_file() and p.suffix in keep_suffixes]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP', zip_path)
print('TEST CSV', OUT_DIR / 'test_results.csv')
print('TEST MD', OUT_DIR / 'test_results.md')
for file in sorted(files):
    print(file)
